# DPS HLS composites
XArray-optimized script

In [1]:
from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

In [11]:
import glob
import datetime
import sys
import shapely
import geopandas as gpd
import pandas as pd
ICESAT2_BOREAL_LIB_PATH = '/projects/code/icesat2_boreal/lib'   
sys.path.append(ICESAT2_BOREAL_LIB_PATH)

In [12]:
res = maap.listAlgorithms()
# this is the correct one
# {'type': 'hls-composite', 'version': 'dev'},
[x for x in res.json()['algorithms'] if 'hls' in x['type']]

[{'type': 'query-hls-stac-geoparquet', 'version': 'main'},
 {'type': 'hls-composite', 'version': 'dev_v1.1'},
 {'type': 'query-hls-stac-geoparquet', 'version': 'debug2'},
 {'type': 'query-hls-stac-geoparquet', 'version': 'credential-chain'},
 {'type': 'hls-composite', 'version': 'dev'},
 {'type': 'hls-landsat-tile', 'version': 'main'},
 {'type': 'query-hls-stac-geoparquet', 'version': 'v0.1'},
 {'type': 'stage1n-hls-extraction', 'version': 'main'},
 {'type': 'query-hls-stac-geoparquet', 'version': 'debug'},
 {'type': 'hls-composite', 'version': 'dev_v1.2'},
 {'type': 'stage1n-hls-batch', 'version': 'main'},
 {'type': 'hls-phenometrics', 'version': 'main'}]

In [13]:
# import json
# r = maap.getQueues()
# r.json()

## Look at previous HLS comps

In [14]:
import mosaiclib

In [15]:
mosaiclib.HLS_TINDEX_FN_DICT

{'c2020oldv1': 's3://maap-ops-workspace/shared/nathanmthomas/DPS_tile_lists/HLS/spring2022/HLS_tindex_master.csv',
 'c2020oldv2': 's3://maap-ops-workspace/shared/nathanmthomas/DPS_tile_lists/HLS/fall2022/HLS_stack_2022_v2/HLS_tindex_master.csv',
 'c2020v2022nmt': 's3://maap-ops-workspace/shared/nathanmthomas/DPS_tile_lists/HLS/c2020/HLS_stack_2022_v2/HLS_tindex_master.csv',
 'c2020v2022pmm': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/HLS/c2020/HLS_stack_2022_v2/HLS_tindex_master.csv',
 'c2020v2022datelines': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/HLS/HLS_stack_2023_v1/HLS_L30_c2020/HLS_tindex_master.csv',
 'c2020v2022updated': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/HLS/c2020/HLS_stack_2022_v2/HLS_tindex_master_updated.csv',
 'c2020v2022updated_bad': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/HLS/HLS_stack_2022_v2/HLS_tindex_master.csv',
 '2015': 's3://maap-ops-workspace/shared/nathanmthomas/DPS_tile_lists/HLS/c2015/HLS_sta

In [16]:
hls_gdf = gpd.read_file(mosaiclib.HLS_TINDEX_FN_DICT['2023'])

In [17]:
hls_gdf.head()

,index,s3_path,local_path,file,tile_num,creation time
0,0,s3://maap-ops-workspace/montesano/dps_output/d...,/projects/my-private-bucket/dps_output/do_HLS_...,HLS_289_06-15_09-15_2023_2023.tif,289,2025-12-01 10:40:34
1,1,s3://maap-ops-workspace/montesano/dps_output/d...,/projects/my-private-bucket/dps_output/do_HLS_...,HLS_234_06-15_09-15_2023_2023.tif,234,2025-09-25 00:14:40
2,2,s3://maap-ops-workspace/montesano/dps_output/d...,/projects/my-private-bucket/dps_output/do_HLS_...,HLS_1322_07-01_08-31_2023_2023.tif,1322,2024-11-08 17:57:23
3,3,s3://maap-ops-workspace/montesano/dps_output/d...,/projects/my-private-bucket/dps_output/do_HLS_...,HLS_1327_07-01_08-31_2023_2023.tif,1327,2024-11-08 17:56:21
4,4,s3://maap-ops-workspace/montesano/dps_output/d...,/projects/my-private-bucket/dps_output/do_HLS_...,HLS_1324_07-01_08-31_2023_2023.tif,1324,2024-11-08 17:54:46


In [18]:
hls_gdf[hls_gdf.tile_num == '3094'].s3_path.to_list()[0]

's3://maap-ops-workspace/montesano/dps_output/do_HLS_stack_3-1-2/HLS_stack_2023_v1/HLS_H30_2023/mc25_07-01_08-31_2023_2023/2023/10/13/14/17/39/130503/HLS_3094_07-01_08-31_2023_2023.tif'

### Read an output file

In [19]:
# fn = '/projects/my-private-bucket/dps_output/do_HLS_stack_3-1-2/HLS_stack_2023_v1/HLS_H30_2023/mc25_07-01_08-31_2023_2023/2023/10/13/14/17/39/130503/_stdout.txt'
# with open(fn, 'r') as file:
#     content = file.read()

# print(content)

# this selects all features in a gpkg 
### if you need to filter just a few tiles, add 

`if row['tile_num'] in [tile_nums of interest]` 

### after the for loop below

In [22]:
tiles_path = 'https://maap-ops-workspace.s3.amazonaws.com/shared/montesano/databank/boreal_tiles_v004_model_ready.gpkg'
gdf = gpd.read_file(tiles_path)

In [ ]:
crs = gdf.crs.to_wkt()
bounds_df = gdf.bounds

In [ ]:
DPS_TILE_LIST = [3829, 2273, 3094]
DPS_TILE_LIST = [3094]

In [24]:
gdf.tile_group.unique()

array(['eurasia west', 'north america east', 'eurasia central',
       'north america centra', 'north america west', 'eurasia east',
       'europe north', 'uk', 'fennoscand south', 'canada extra',
       'eurasia north', 'conus extra', 'alaska extra'], dtype=object)

In [32]:
DPS_TILE_LIST = gdf[gdf.tile_group == 'north america west'].tile_num.to_list()[0:25]
len(DPS_TILE_LIST)

25

In [33]:
filtered = gdf[gdf['tile_num'].isin(DPS_TILE_LIST)].join(gdf.bounds)

feats = [
    {
        'tile_num': row['tile_num'],
        'bounds': (row['minx'], row['miny'], row['maxx'], row['maxy']),
        'bounds_str': f"{row['minx']} {row['miny']} {row['maxx']} {row['maxy']}",
        'dims': ((row['maxx'] - row['minx']) / 30, (row['maxy'] - row['miny']) / 30),
    }
    for _, row in filtered.iterrows()
]

print(len(feats))
feats[0]

25


{'tile_num': 2124,
 'bounds': (2528521.9999999953,
  4473304.000000009,
  2618521.9999999953,
  4563304.000000009),
 'bounds_str': '2528521.9999999953 4473304.000000009 2618521.9999999953 4563304.000000009',
 'dims': (3000.0, 3000.0)}

In [34]:
# Make sure they are all 3000x3000
{d['dims'] for d in feats}

{(3000.0, 3000.0)}

### `TODO`
2. why so many dark pixels from median comps? - must change params - control min cloud and max n params in DPS notebook and use those to set DPS_IDENTIFIER
3. want filename like: HLS_1841_07-01_08-31_2024_2024_q050ndvi.tif

In [35]:
MAX_CC = 50
MAX_N = 200

In [36]:
#index_options = ['ndvi', 'savi', 'msavi', 'evi', 'evi2', 'ndmi', 'ndsi', 'nbr', 'nbr2']

def get_input_params_dict(feat, year, start_month=7, end_month=8):
    return {
        'start_datetime': f"{year}-{start_month:02d}-01T00:00:00Z",
        'end_datetime': f"{year}-{end_month:02d}-31T00:00:00Z",
        'bbox': feat['bounds_str'],
        'crs': crs,
        'output_name': f'HLS_{feat['tile_num']}_{start_month:02d}-01_{end_month:02d}-31_{year}_{year}.tif',
        'composite_type': 'qndvi', # quantile-ndvi
        'q': '0.50', # 50th percentile (median NDVI composite)
        'aoi': '', # leave blank, handy to pass single_feature.gpkg for local development
        #'lim': 200, # max number of items to composite, might need bigger queue if set > 200
        'max_n': MAX_N,
        'max_cloud_cover': MAX_CC,
        'catalog': '', # leave blank, not working
        'indices': '', # see index_options obove, for more indexes pass e.g 'nbr ndvi evi'
        'count_layers': 'raw_count strict_count relaxed_count final_count'
    }

ALGO_ID = "hls-composite"
USER = "" # <---------------------------------------------- update ----------------------------------------
WORKER_TYPE = 'maap-dps-worker-16gb' 
VERSION =  "dev_v1.2"

### Set up a dir to hold the metadata output table from the DPS submission

In [37]:
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results/'
!mkdir -p $DPS_SUBMISSION_RESULTS_DIR

# submit all tiles in feats list, all years

### set the identifier and run-name below

In [38]:
%%time
submit_results_df_list = []

for YEAR in range(2016, 2026):
    
    IDENTIFIER = f'HLS_H30_{YEAR}/mc{MAX_CC:03}_mn{MAX_N:03}_07-01_08-31_{YEAR}_{YEAR}'
    RUNNAME = f'{IDENTIFIER}'.replace('/', '_')
    for i, feat in enumerate(feats):
        DPS_num = i+1
        try:
            in_params_dict = get_input_params_dict(feat, YEAR)
            submit_result = maap.submitJob(
                    identifier=IDENTIFIER,
                    algo_id=ALGO_ID,
                    version=VERSION,
                    username=USER,
                    queue=WORKER_TYPE,
                    **in_params_dict
            )
            submit_result_df = pd.DataFrame( 
                {
                        'dps_num':[DPS_num],
                        'tile_num':[feat['tile_num']],
                        'year': [YEAR],
                        'submit_time':[datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%s')],
                        'dbs_job_hour': [datetime.datetime.now().hour],
                        'algo_id': [ALGO_ID],
                        'user': [USER],
                        'worker_type': [WORKER_TYPE],
                        'job_id': [submit_result.id],
                        'submit_status': [submit_result.status],
                    
                } 
            )        
            submit_results_df_list.append(submit_result_df)
            if DPS_num in [1, 5, 10, 50, 100, 250, 500, 750, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 7000, 9000, 11000, 13000, 15000, 17000, 19000, 21000, 24000, len(feats)]:
                print(f"DPS run #: {DPS_num}\t| tile num: {feat['tile_num']}\t| submit status: {submit_result.status}\t| job id: {submit_result.id}") 
    
        except Exception as e:
            print(f'skipping {feat['tile_num']}')
            print(f'due to {e}')
        
        
# Build a final submission results data frame and save
submit_results_df = pd.concat(submit_results_df_list)
submit_results_df['run_name'] = IDENTIFIER
nowtime = pd.Timestamp.now().strftime('%Y%m%d%H%M')
print(f"Current time:\t{nowtime}")

# This creates a CSV of the metadata associated with the DPS jobs you have just submitted
submit_results_df.to_csv(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{RUNNAME}_submission_results_{len(feats)}_{nowtime}.csv')
submit_results_df.info()

DPS run #: 1	| tile num: 2124	| submit status: success	| job id: cf98519a-b767-402e-9fa0-6618b23e5f78
DPS run #: 5	| tile num: 2272	| submit status: success	| job id: 24cc206c-c28c-4eea-86aa-520c79175ecd
DPS run #: 10	| tile num: 2348	| submit status: success	| job id: f0416522-2b6f-4589-89a5-3b4c3b8f29cf
DPS run #: 25	| tile num: 2434	| submit status: success	| job id: d34facb4-404f-4199-ae2e-e33da55cdb5f
DPS run #: 1	| tile num: 2124	| submit status: success	| job id: 8f52f1d6-62a3-4463-8829-ece8143e1126
DPS run #: 5	| tile num: 2272	| submit status: success	| job id: dcb1dece-7230-41ec-8a2f-a67ed30b2a45
DPS run #: 10	| tile num: 2348	| submit status: success	| job id: 26a667fe-1185-42c5-86cf-02ebcd18b0c2
DPS run #: 25	| tile num: 2434	| submit status: success	| job id: 1a8bc63a-1d2f-4c5a-955a-451183742b32
DPS run #: 1	| tile num: 2124	| submit status: success	| job id: 06d1339a-40a2-4da8-869d-780118ce8a57
DPS run #: 5	| tile num: 2272	| submit status: success	| job id: 76f5d58f-c13d

In [26]:
RUNNAME,submit_results_df

('test_mc90_mn200_07-01_08-31_2025_2025',
    dps_num  tile_num  year                  submit_time  dbs_job_hour  \
 0        1      2273  2016  2026-06-09-15-34-1781044444            15   
 0        2      3094  2016  2026-06-09-15-34-1781044444            15   
 0        3      3829  2016  2026-06-09-15-34-1781044444            15   
 0        1      2273  2017  2026-06-09-15-34-1781044444            15   
 0        2      3094  2017  2026-06-09-15-34-1781044444            15   
 0        3      3829  2017  2026-06-09-15-34-1781044444            15   
 0        1      2273  2018  2026-06-09-15-34-1781044444            15   
 0        2      3094  2018  2026-06-09-15-34-1781044444            15   
 0        3      3829  2018  2026-06-09-15-34-1781044445            15   
 0        1      2273  2019  2026-06-09-15-34-1781044445            15   
 0        2      3094  2019  2026-06-09-15-34-1781044445            15   
 0        3      3829  2019  2026-06-09-15-34-1781044445            15

After almost any DPS job, you have to assess what was marked as `success` and `fail`.  

This involves:
1. building a table of job status based on job ids captured in the job_results_df from the DPS run chunk (this takes 40 mins for ~47k jobs). This tells you how many jobs failed.
2. merging the `job status table` with the `job results df`. This tells you which specific granules (or tile nums) failed.
3. building another input list of granules/tiles for a follow-up DPS run.
## Assess DPS results
Build a table of job status based on job id - how many jobs failed?

In [27]:
from pathlib import Path
import importlib
import sys

ICESAT2_BOREAL_REPO_PATH = str(Path('/projects/code/icesat2_boreal/lib/').expanduser())
sys.path.append(ICESAT2_BOREAL_REPO_PATH)

import ExtractUtils
importlib.reload(ExtractUtils)

NASA MAAP
NASA MAAP


<module 'ExtractUtils' from '/projects/code/icesat2_boreal/lib/ExtractUtils.py'>

In [28]:
DPS_SUBMISSION_RESULTS_DIR
#ALGO_ID
#glob.glob(f'/projects/{DPS_SUBMISSION_RESULTS_DIR}/*')

'/projects/my-public-bucket/dps_submission_results/'

In [29]:
LIST_SUBMISSIONS = sorted(glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_*_submission_results_*.csv'),key=ExtractUtils.func, reverse=True)
LIST_SUBMISSIONS[:5]

['/projects/my-public-bucket/dps_submission_results/DPS_hls-composite_test_mc90_mn200_07-01_08-31_2025_2025_submission_results_3_202606091534.csv']

In [30]:
%%time

running_list = []
fails_list = []
success_list = []
offline_list = []

for fn in LIST_SUBMISSIONS[:1]:
    print(fn)
    df_jstatus = ExtractUtils.BUILD_TABLE_JOBSTATUS(pd.read_csv(fn))
    df_jstatus['run_type'] = ALGO_ID
    running_list.append(df_jstatus[ (df_jstatus['status'] == 'Running') ] )
    fails_list.append(  df_jstatus[ (df_jstatus['status'] == 'Failed') ] )
    success_list.append(df_jstatus[ (df_jstatus['status'] == 'Succeeded') ] )
    offline_list.append(df_jstatus[ (df_jstatus['status'] == 'Offline') ] )
    print(f"Count offline jobs: {df_jstatus[ (df_jstatus['status'] == 'Offline') ].shape[0]}\n")
            
df_all_running = pd.concat(running_list)          
df_all_fails =   pd.concat(fails_list)
df_all_success = pd.concat(success_list)
df_all_offline = pd.concat(offline_list)

/projects/my-public-bucket/dps_submission_results/DPS_hls-composite_test_mc90_mn200_07-01_08-31_2025_2025_submission_results_3_202606091534.csv
multiprocessing...
Count total jobs:	30
Count pending jobs:	2
Count running jobs:	28
Count succeeded jobs:	0
Count failed jobs:	0
% of failed jobs:	Nothing has failed...yet

Count offline jobs: 0

CPU times: user 25.9 ms, sys: 137 ms, total: 163 ms
Wall time: 515 ms


In [17]:
failed_or_offline = df_all_fails.tile_num.to_list() + df_all_offline.tile_num.to_list()
print(len(failed_or_offline))
failed_or_offline[:10]

0


[]

In [ ]:
maap.getJob('8b9c4e34-0dae-4545-90d9-4ee834c9d4a1')

# This creates a tindex

In [5]:
import boto3

def get_tindex(year):

    def get_info(key):
        name = Path(key).name
        parts = name.split('_')
        info = dict()
        info['s3_path'] = 's3://maap-ops-workspace/' + key
        info['tile_num'] = parts[1]
        info['year'] = parts[4]
        info['timestamp'] = datetime.datetime.strptime(''.join(key.split('/')[7:13]), format_code)
        return info
    # if year == 2021:
    #     prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}_fixed/relaxed_mask/mc90_mn200_07-01_08-31_{year}_{year}'
    # else:
    #     prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}/relaxed_mask/mc90_mn200_07-01_08-31_{year}_{year}'
    #prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}_/AK_expand3/mc90_mn200_07-01_08-31_{year}_{year}'
    #prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}/CA_22_burnt_tiles/mc90_mn200_07-01_08-31_{year}_{year}'
    #prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}/five_tiles_around_3364/mc90_mn200_07-01_08-31_{year}_{year}'
    prefix = f'aliz237/dps_output/hls-composite/dev/HLS_H30_{year}/ak_expand4/mc90_mn200_07-01_08-31_{year}_{year}'
    get_info = get_info
    search_key = '.tif'
    

    format_code = "%Y%m%d%H%M%S%f"
    
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(
        Bucket='maap-ops-workspace',
        Prefix=prefix
    )

    tindex = []
    for page in page_iterator:
        try:
            for item in page['Contents']:
                key = item.get('Key', '')
                if key.endswith(search_key):
                    data = get_info(key)
                    tindex.append(data)
        except KeyError as e:
            print(e)
            print(f'No Contents, most likely the s3 prefix {prefix} is wrong')
            raise
    
    tindex = pd.DataFrame(tindex).sort_values('timestamp').groupby(['tile_num', 'year']).last().reset_index()
    return tindex



for year in range(2019, 2025):
    t = get_tindex(year)
    # t.head(), 
    print(f'year:{year}, n:{t.tile_num.unique().shape}')
    #t.to_csv(f'/home/jovyan/Download/Alaska/ak_hls_tindex_{year}_fixed_relaxed_mask.csv', index=False)
    #t.to_csv(f'/home/jovyan/data/AK/tindex/ak_hls_tindex_5_tiles_3364_{year}.csv', index=False)
    t.to_csv(f'/home/jovyan/data/AK/tindex/ak_hls_tindex_{year}_expand4.csv', index=False)

year:2019, n:(156,)
year:2020, n:(156,)
year:2021, n:(156,)
year:2022, n:(156,)
year:2023, n:(156,)
year:2024, n:(156,)
